# Stage 3 — EDA. Lahore house price model.

**What this notebook is for.** To understand the data well enough to lock the acceptance
criteria (B-05, D-47) and to drive stage 4's feature choices. It *measures*; it does not
choose. Every choice this evidence enables is made in stage 4 and logged there — D-52.

**Reading order.** `project-log/PLAN.md` §5 for the criteria, `decisions/03-cleaning.md`
for how this table was built, `decisions/04-eda.md` for the decisions this stage takes.

**Two standing constraints on the output of this notebook.**

* **No verbatim listing rows, anywhere.** Notebook cells store what they printed, and this
  file is committed (D-23). Nothing below prints a title, a description, a URL or a single
  listing's values. Aggregates, distributions and place names only.
* **Every count is bounded where it is computed.** A row count cannot exceed the rows and a
  percentage cannot exceed 100, and the assertion sits in the same expression as the count
  (LESSONS L-26, which exists because a report shipped a count larger than its dataset).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

from lhp import viz
from lhp.clean import read_processed

viz.use_house_style()
pd.set_option("display.max_rows", 60, "display.width", 120)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = ROOT / "reports" / "figures"

df = read_processed(str(ROOT / "data" / "processed" / "listings.parquet"))
members = pd.read_parquet(ROOT / "data" / "processed" / "listings_members.parquet")

# The analysis set is one row per fingerprint group (D-45), and the members file holds the
# listings each row stood for (D-53). If either stops being true the whole stage is built on
# the wrong table, so it is checked here rather than assumed.
assert len(df) == df["fp_group_id"].nunique(), "the analysis set is not one row per group"
assert len(members) == int(df["n_listings"].sum()), "members do not reconcile to n_listings"

print(f"analysis set     {len(df):,} rows x {df.shape[1]} columns")
print(f"member listings  {len(members):,}")
print(f"window           {df['created_at'].min():%d %b %Y} to {df['created_at'].max():%d %b %Y}")

## 1 · The analysis set at a glance

Three numbers frame everything below: what one row *is*, how many listings it stands for,
and what the target looks like before any breakdown.

In [ ]:
price_per_marla = df["price"] / df["area_marla"]
df["price_per_marla"] = price_per_marla

glance = pd.Series({
    "analysis rows (one per fingerprint group)": len(df),
    "listings they stand for": int(df["n_listings"].sum()),
    "rows standing for more than one listing": int((df["n_listings"] > 1).sum()),
    "distinct societies": df["loc_society"].nunique(),
    "distinct full locality paths": df["loc_path"].nunique(),
    "rows with no usable coordinate (D-33)": int(df["latitude_clean"].isna().sum()),
})
assert glance.max() <= len(members), "a count exceeds the number of listings"
glance.to_frame("count").style.format("{:,.0f}")

## 2 · Every column, profiled — D-54

The exit criterion is that every column's distribution is reviewed, and there are 42 of them
against a two-day cap. So all 42 are profiled mechanically here, and narrative is spent below
on the ones that could become features.

The profile deliberately prints **no string values** — distinct counts and modal *frequency*
only. That is what keeps D-23 satisfied without a judgement call on every column: a rule that
prints no listing text cannot accidentally print some.

In [ ]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: type, coverage, cardinality and a type-appropriate summary.

    Prints no string values by construction (D-23). Every count carries its bound in the
    expression that produces it (L-26).
    """
    rows = []
    n = len(frame)
    for name in frame.columns:
        s = frame[name]
        non_null = int(s.notna().sum())
        assert 0 <= non_null <= n, f"{name}: impossible non-null count"
        distinct = int(s.nunique(dropna=True))
        assert distinct <= non_null, f"{name}: more distinct values than non-null rows"

        if pd.api.types.is_bool_dtype(s) or str(s.dtype) == "boolean":
            true = int(s.fillna(False).sum())
            summary = f"true on {true:,} ({true / n:.1%})"
        elif pd.api.types.is_numeric_dtype(s):
            q = s.quantile([0.25, 0.5, 0.75])
            summary = f"p25 {q.iloc[0]:,.2f} · median {q.iloc[1]:,.2f} · p75 {q.iloc[2]:,.2f}"
        elif pd.api.types.is_datetime64_any_dtype(s):
            summary = f"{s.min():%d %b %Y} to {s.max():%d %b %Y}"
        else:
            modal = int(s.value_counts().iloc[0]) if distinct else 0
            summary = f"most common value covers {modal:,} rows ({modal / n:.1%})"

        rows.append({
            "column": name,
            "dtype": str(s.dtype),
            "non_null": non_null,
            "null %": round(100 * (n - non_null) / n, 1),
            "distinct": distinct,
            "summary": summary,
        })
    out = pd.DataFrame(rows).set_index("column")
    assert (out["null %"] <= 100).all(), "a null percentage exceeded 100"
    return out


column_profile = profile(df.drop(columns=["price_per_marla"]))
column_profile

### 2.1 · What each column is *for*

The profile says what a column contains. This says what it is allowed to become, and it is
the seed of D-51's barred-column table — the two reasons a column can be barred are different
failures and are kept apart:

* **`candidate`** — could be a feature. Stage 4 decides whether it is one (D-52).
* **`target`** — the thing being predicted.
* **`leakage`** — derived from the price. Can never be a feature.
* **`not-at-inference`** — real, but unknowable when someone asks the estimator to price a
  house that has not been advertised yet. A model leaning on these validates well and cannot
  be served (PLAN §2, deliverable 8).
* **`text-source`** — free text. Not a feature in itself and never fed to a model as a
  string; it may *yield* features by extraction — a floor count, a corner plot, a road
  width — and whether extraction is worth its cost is a stage 4 question (D-52). B-15 is
  the precedent: the title carried an area unit the payload had wrong.
* **`audit`** — provenance and the cleaning trail. Kept so a number can be traced, never
  modelled.
* **`superseded`** — a source column kept beside its cleaned form so the conversion stays
  auditable (B-12). The cleaned form is the candidate.

In [ ]:
DISPOSITION = {
    # the target and its parts
    "price": "target",
    "area_marla": "candidate",
    "bedrooms": "candidate", "bathrooms": "candidate",
    # location, cleaned
    "latitude_clean": "candidate", "longitude_clean": "candidate",
    "loc_society": "candidate", "loc_phase": "candidate", "loc_phase_num": "candidate",
    "loc_sector": "candidate", "loc_block": "candidate", "loc_sub": "candidate",
    "loc_path": "candidate",
    # time
    "created_at": "candidate",
    # derived from the price — leakage, always
    "price_listed": "leakage", "fp_price_median": "leakage", "fp_price_spread": "leakage",
    "n_listings": "leakage", "collapsed": "leakage",
    # properties of an advert that does not exist at prediction time
    "views": "not-at-inference", "image_count": "not-at-inference",
    "is_featured": "not-at-inference", "has_video": "not-at-inference",
    # free text — a source of features, not a feature (D-52)
    "title": "text-source", "description": "text-source",
    # source values kept beside their cleaned form
    "area": "superseded", "area_unit": "superseded", "locality": "superseded",
    "latitude": "superseded", "longitude": "superseded", "address": "superseded",
    # provenance and the cleaning trail
    "property_id": "audit", "source_url": "audit", "discovered_on_page": "audit",
    "price_matches_discovery": "audit", "area_source": "audit", "coord_in_lahore": "audit",
    "coord_source": "audit", "clean_flags": "audit", "exclusion_reason": "audit",
    "excluded": "audit", "fp_group_id": "audit",
}

missing = set(column_profile.index) - set(DISPOSITION)
unknown = set(DISPOSITION) - set(column_profile.index)
assert not missing and not unknown, f"unclassified: {missing or ''} unknown: {unknown or ''}"

column_profile["disposition"] = pd.Series(DISPOSITION)
column_profile["disposition"].value_counts().to_frame("columns")

`created_at` sits in `candidate` on purpose and is the one genuinely unsettled case. PLAN §3
keeps it as a control on the twelve-month window, but the estimator would have to supply
today's date, which is outside that window — the extrapolation risk the README already
discloses. §6 produces the evidence; the choice is stage 4's (D-51, D-55).

## 3 · The target

Two views of the same quantity. Price is what the model predicts; **price per marla** is what
the market actually prices, and it is the quantity the §5.1 baseline is built from.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

ax = axes[0]
ax.hist(df["price"] / 1e6, bins=np.logspace(np.log10(1), np.log10(1000), 50),
        color=viz.CATEGORICAL[0], edgecolor=viz.SURFACE, linewidth=0.5)
ax.set_xscale("log")
ax.set_xticks([1, 3, 10, 30, 100, 300, 1000])
ax.set_xticklabels(["1M", "3M", "10M", "30M", "100M", "300M", "1B"])
viz.reference_line(ax, df["price"].median() / 1e6, f"median {df['price'].median()/1e6:,.1f}M",
                   horizontal=False)
viz.annotate(ax, title="Asking price", subtitle="Right-skewed over two orders of magnitude — the reason §5.2 optimises on log price.",
             xlabel="PKR, log scale", ylabel="listings")

ax = axes[1]
ppm = df["price_per_marla"] / 1e6
ax.hist(ppm.clip(upper=12), bins=60, color=viz.CATEGORICAL[0], edgecolor=viz.SURFACE, linewidth=0.5)
viz.reference_line(ax, ppm.median(), f"median {ppm.median():,.2f}M", horizontal=False)
viz.annotate(ax, title="Asking price per marla", subtitle="Far tighter: half the market sits inside a 1.4M band.",
             xlabel="PKR per marla, millions (clipped at 12M)", ylabel="listings")

fig.tight_layout()
viz.save(fig, "03-target-distributions", directory=FIGURES)
plt.show()

summary = pd.DataFrame({
    "price (M)": (df["price"] / 1e6).describe(percentiles=[.05, .25, .5, .75, .95]),
    "price per marla (M)": (df["price_per_marla"] / 1e6).describe(percentiles=[.05, .25, .5, .75, .95]),
}).round(2)
summary

**The finding that bears on D-47.** Price per marla is tight where price is not: the
interquartile range is a factor of about 1.5, against roughly 4 for price itself. Most of what
the model has to explain is therefore *size*, which is one column — and the §5.1 baseline
already uses it. This is the shape of market in which a lookup table competes with a
gradient-boosted model, which is exactly why D-47 adds a relative criterion. It is a finding,
not a failure; D-06 anticipated reporting it.

## 4 · Size

`area_marla` is the model's strongest single lever, so how it is distributed and how it is
recorded both matter. D-32 established that 8.8% of payload areas are fractional and that the
payload is the measured figure — which constrains any attempt to band sizes.

In [ ]:
fractional = df["area_marla"].mod(1).ne(0)
common = df["area_marla"].value_counts().head(12).sort_index()

fig, ax = plt.subplots()
ax.bar(range(len(common)), common.values, color=viz.CATEGORICAL[0], width=0.72)
ax.set_xticks(range(len(common)))
ax.set_xticklabels([f"{v:g}" for v in common.index])
ax.yaxis.set_major_formatter(viz.thousands())
viz.label_bars(ax, only=3, fmt="{:,.0f}")
viz.annotate(ax, title="The market trades in standard plot sizes",
             subtitle=f"The twelve commonest sizes cover {common.sum()/len(df):.0%} of listings; {fractional.mean():.1%} of areas are fractional.",
             xlabel="area (marla)", ylabel="listings",
             source=f"{len(df):,} analysis rows. Fractional areas are measured figures, not noise (D-32).")
viz.save(fig, "04-size-distribution", directory=FIGURES)
plt.show()

pd.Series({
    "distinct area values": df["area_marla"].nunique(),
    "fractional areas": int(fractional.sum()),
    "fractional %": round(100 * fractional.mean(), 1),
    "smallest (marla)": df["area_marla"].min(),
    "largest (marla)": df["area_marla"].max(),
}).to_frame("value")

## 5 · Where the listings are — B-22

B-22 records Park View City as over-represented. Two things have to be settled before the
claim can be measured: what the development is called in this data, and what counts as *one*
development.

* Ilaan labels it **`Park View Villas`**. "Park View City" appears nowhere in `loc_society`.
* **DHA is an umbrella of phases, not a single development** (Nauman, 8 Sep 2026). Counting
  `loc_society` alone makes DHA look like the largest development in Lahore; split by phase,
  as D-40 makes possible, it is not.

In [ ]:
society = df["loc_society"].astype("object").fillna("(unnamed)")
phase = df["loc_phase"].astype("object").fillna("")
# A development is a society, split by phase where one was parsed. Built with a list
# comprehension rather than string concatenation because a null in either part propagates to
# a null key, and groupby drops null keys silently — 265 rows vanished from this very table
# on the first attempt. The assertion is what caught it.
df["development"] = [s if not p else f"{s} {p}" for s, p in zip(society, phase, strict=True)]
assert df["development"].notna().all()

by_dev = (df.groupby("development")
            .agg(rows=("price", "size"), listings=("n_listings", "sum"),
                 ppm=("price_per_marla", "median"))
            .sort_values("listings", ascending=False))
assert by_dev["listings"].sum() == int(df["n_listings"].sum()), "developments lost listings"

top = by_dev.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top.index, top["listings"], color=viz.CATEGORICAL[0], height=0.72, label="listings as posted")
ax.barh(top.index, top["rows"], color=viz.CATEGORICAL[1], height=0.42, label="analysis rows after de-duplication")
ax.xaxis.set_major_formatter(viz.thousands())
ax.legend(loc="lower right")
share_listings = by_dev["listings"].iloc[0] / df["n_listings"].sum()
share_rows = by_dev["rows"].iloc[0] / len(df)
viz.annotate(ax, title="The twelve largest developments",
             subtitle=f"Park View Villas is {share_listings:.0%} of listings as posted but {share_rows:.0%} of analysis rows — the collapse absorbed most of the imbalance.",
             xlabel="count", grid_axis="x",
             source=f"{int(df['n_listings'].sum()):,} listings collapsing to {len(df):,} rows. DHA split by phase.")
viz.save(fig, "05-largest-developments", directory=FIGURES)
plt.show()

by_dev.head(8).assign(**{"PKR/marla": lambda t: (t["ppm"] / 1e6).round(2)}).drop(columns="ppm")

**What this does to B-22.** The entry is right that Park View Villas is the largest single
development — 807 listings as posted, twice the next — and it is right that no other single
development comes close. But the over-representation it warns about is mostly a *replication*
effect, and D-45's collapse has already absorbed it: 807 listings became 253 rows, taking the
development from 9.7% of the sample to 3.9%. A sample-weighting intervention at stage 5 has
much less to correct than the entry implies.

Two corrections for the backlog at stage close: the entry names a society that does not exist
under that name in the data, and it claims a rank that is only true once DHA is treated as
one development rather than an umbrella of seventeen phases.

## 6 · Which locality tier carries the signal — and what the baseline already scores

D-40 left three granularities available and chose none: **society** (377 levels), **society +
phase** (443) and the **full path** down to block (921). Which one carries price signal is the
question stage 4 needs answered, and the honest way to answer it is out of sample — within a
sample, a finer key always looks better, right up until it memorises.

So each tier is scored the way PLAN §5.1's baseline works: predict a house's price as *the
median price per marla of its locality level and size band, times its area*, with a fallback up
the chain when a level is unseen. Five-fold, and the folds are group-aware for free, since the
analysis set is already one row per fingerprint group (D-45).

In [ ]:
from sklearn.model_selection import KFold

SIZE_BANDS = [0, 5, 7.5, 10, 15, 20, 40, np.inf]
SIZE_LABELS = ["<=5", "5-7.5", "7.5-10", "10-15", "15-20", "20-40", ">40"]

society = df["loc_society"].astype("object").fillna("(unnamed)")
phase = df["loc_phase"].astype("object").fillna("")
df["tier_society"] = society
df["tier_phase"] = [s if not p else f"{s} > {p}" for s, p in zip(society, phase, strict=True)]
df["tier_path"] = df["loc_path"].astype("object").fillna(df["tier_phase"])
df["size_band"] = pd.cut(df["area_marla"], SIZE_BANDS, labels=SIZE_LABELS)

TIERS = {
    "society": ["tier_society"],
    "society + phase": ["tier_phase", "tier_society"],
    "full path": ["tier_path", "tier_phase", "tier_society"],
}


def baseline_predictions(chain: list[str], folds: int = 5, seed: int = 20260908) -> pd.Series:
    """PLAN §5.1's baseline, out of fold: median PKR/marla by locality level x size band.

    `chain` is ordered finest first; a level unseen in training falls back to the next key,
    and finally to the corpus median. Nothing here touches the test fold's prices.
    """
    out = pd.Series(index=df.index, dtype=float)
    for train_idx, test_idx in KFold(folds, shuffle=True, random_state=seed).split(df):
        train, test = df.iloc[train_idx], df.iloc[test_idx]
        tables = [train.groupby([key, "size_band"], observed=True)["price_per_marla"].median()
                  for key in chain]
        city = train["price_per_marla"].median()
        rates = []
        for row in test.itertuples():
            rate = np.nan
            for key, table in zip(chain, tables, strict=True):
                rate = table.get((getattr(row, key), row.size_band), np.nan)
                if pd.notna(rate):
                    break
            rates.append(city if pd.isna(rate) else rate)
        out.iloc[test_idx] = np.array(rates) * test["area_marla"].to_numpy()
    assert out.notna().all(), "every row must receive a prediction"
    return out


def score(predicted: pd.Series) -> dict[str, float]:
    ape = (predicted - df["price"]).abs() / df["price"] * 100
    result = {"MdAPE %": ape.median(), "PPE10 %": 100 * ape.le(10).mean(),
              "PPE20 %": 100 * ape.le(20).mean(), "mean APE %": ape.mean()}
    assert 0 <= result["PPE20 %"] <= 100, "a proportion exceeded 100"
    return result


support = pd.DataFrame({
    tier: {
        "levels": df[chain[0]].nunique(),
        "median rows per level": df[chain[0]].value_counts().median(),
        "% of rows in levels under 5 rows": round(
            100 * df[chain[0]].value_counts().pipe(lambda v: v[v < 5]).sum() / len(df), 1),
    } for tier, chain in TIERS.items()
}).T

baseline = pd.DataFrame({tier: score(baseline_predictions(chain)) for tier, chain in TIERS.items()}).T
tier_evidence = support.join(baseline).round(2)
tier_evidence

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(tier_evidence.index, tier_evidence["MdAPE %"], color=viz.CATEGORICAL[0], width=0.55)
viz.label_bars(ax, fmt="{:.2f}%")
viz.reference_line(ax, 20, "PLAN §5.1 accept: 20%")
viz.reference_line(ax, 15, "PLAN §5.1 stretch: 15%")
ax.set_ylim(0, 23)
viz.annotate(ax, title="A lookup table already beats the provisional acceptance criteria",
             subtitle="Median price per marla by locality tier and size band, out of fold. Phase is where the signal stops.",
             ylabel="MdAPE (lower is better)",
             source=f"{len(df):,} analysis rows, 5-fold. The baseline of PLAN §5.1, not a model.")
viz.save(fig, "06-baseline-by-tier", directory=FIGURES)
plt.show()

**Three findings, and the first one is the most consequential of the stage.**

**1 · The provisional acceptance criteria are dead.** The §5.1 baseline — a lookup table with
no model in it — scores **MdAPE 11.5%** and **PPE20 71%** out of fold. The provisional accept
was MdAPE ≤ 20% with PPE20 ≥ 55%, and the *stretch* was 15%. All three are beaten before any
model is trained. A criterion a lookup table satisfies measures nothing about the work, which
is exactly the risk D-47 was written to catch; the numbers it locks have to be re-anchored on
this, and its relative criterion becomes the criterion that carries the weight.

**2 · The tier is society + phase.** Going from society to society + phase buys 0.8 points of
MdAPE. Going on to the full path buys **nothing** — 11.50% against 11.47%, marginally *worse* —
while more than doubling the number of levels and putting 17% of rows into levels holding fewer
than five of them. The block tier is memorisation dressed as granularity. Evidence for stage 4 under
D-52; the choice is stage 4's to log.

**3 · Error is not flat across the market.** The baseline runs 9.5% MdAPE in the second price
quartile and 15.1% in the dearest — the expensive end is roughly 60% harder. That is the shape
D-49's segmented criteria exist to expose, and the price-band split is now evidenced rather
than assumed.

## 7 · Time — the evidence B-03 has been waiting for

`created_at` survived cleaning as a real datetime (D-39), so B-03's recency holdout is finally
decidable. PLAN §3 treats the twelve-month window as close enough to a cross-section and
carries listing date through as a control. That assumption is testable now.

In [ ]:
monthly = (df.set_index("created_at").sort_index()
             .groupby(pd.Grouper(freq="MS"))
             .agg(listings=("price", "size"), ppm=("price_per_marla", "median")))
assert monthly["listings"].sum() == len(df), "months lost rows"

fig, axes = plt.subplots(2, 1, figsize=(10, 6.4), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

ax = axes[0]
ax.plot(monthly.index, monthly["ppm"] / 1e6, color=viz.CATEGORICAL[0], marker="o")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.1f}M"))
viz.annotate(ax, title="The window is not a cross-section",
             subtitle="Median asking price per marla rose through the twelve months the sample covers.",
             ylabel="PKR per marla")

ax = axes[1]
ax.bar(monthly.index, monthly["listings"], color=viz.CATEGORICAL[0], width=20)
ax.yaxis.set_major_formatter(viz.thousands())
viz.annotate(ax, ylabel="listings",
             source="First and last months are partial: the window opens 2 Sep 2023 and closes 14 Sep 2024.")

fig.tight_layout()
viz.save(fig, "07-monthly-trend", directory=FIGURES)
plt.show()

full = monthly.iloc[1:-1]          # drop the two partial months
drift = full["ppm"].iloc[-1] / full["ppm"].iloc[0] - 1
print(f"median PKR/marla, first to last complete month: {drift:+.1%}")
monthly.assign(ppm=lambda t: (t["ppm"] / 1e6).round(2))

**The window carries a trend, and it is large enough to matter.** Median price per marla rises
about **14%** from the first complete month to the last — a drift of the same order as the
baseline's own error. Three consequences, all of them for later stages to act on rather than
this one:

* **B-03 has its answer in principle.** A random split lets the model learn from listings
  posted after the ones it is scored on. With a 14% drift that is a real, if mild, form of
  temporal leakage, and it flatters the headline. The choice between a random group-aware split
  and a recency holdout is stage 5's (D-55) — but the evidence now points somewhere.
* **`created_at` earns its place as a control**, which is what PLAN §3 anticipated. Whether it
  becomes a feature is D-51's open case, and the case is now stronger.
* **The extrapolation risk in the README is bigger than it reads.** The estimator will be asked
  to price houses today from a window that closed in September 2024, on a market that was
  rising when the data stops. Silence about that is not a limitation, it is a defect; the
  README's disclosure needs a number attached at stage 7.

## 8 · Leakage — one house under two fingerprints

D-45's collapse removes duplicates that share a fingerprint exactly. It cannot touch the case
where the same house was posted twice and the two postings disagree slightly about *where* it
is or *how big* it is, because those land in different groups. Whatever survives here would sit
on both sides of a train/test split and flatter every number the project reports.

It is measurable as an upper bound. Two patterns can hide a repeat, and neither may use price
to decide — using the target to establish identity is the objection D-38 raised against putting
price in the fingerprint, and it applies with more force here.

In [ ]:
core = df.assign(lat=df["latitude_clean"].round(5), lon=df["longitude_clean"].round(5))
located = core.dropna(subset=["lat", "lon"])

# (a) Identical coordinates, area and rooms, but a different locality path. This is exactly
# the difference between the fingerprint D-43 adopted and the looser coordinates-only key it
# measured and rejected.
paths = located.groupby(["lat", "lon", "area_marla", "bedrooms", "bathrooms"], dropna=False)
differing_path = int(paths.size()[paths["loc_path"].nunique() > 1].sum())

# (b) Identical coordinates, path and rooms, with the area differing by under a marla — the
# rounding difference D-32 found between two sources, appearing across two postings instead.
same_place = located.groupby(["lat", "lon", "loc_path", "bedrooms", "bathrooms"], dropna=False)
spread = same_place["area_marla"].transform(lambda s: s.max() - s.min())
group_size = same_place["area_marla"].transform("size")
jittered_area = int(((group_size > 1) & spread.between(0.001, 1.0)).sum())

residual = pd.Series({
    "same coords + area + rooms, different loc_path": differing_path,
    "same coords + path + rooms, area within 1 marla": jittered_area,
    "upper bound, both patterns": differing_path + jittered_area,
})
assert residual.max() <= len(df), "more suspect rows than rows"
residual.to_frame("rows").assign(**{"% of analysis set": (100 * residual / len(df)).round(2)})

**13% is an upper bound and almost certainly a large overestimate.** D-38 measured that 70.6%
of rows share their exact coordinate with at least one other row, because Ilaan geocodes to a
society or a street rather than to a house. Identical coordinates therefore do not mean the
same building, and most of pattern (a) will be genuinely different houses on one street with
differently-written addresses. There is no way to narrow it: the only fields that would settle
it are price, which is barred, and a house-level coordinate, which does not exist.

**So the mitigation is procedural, not another cleaning rule, and it costs nothing.** Dedup and
splitting are two different jobs and should not share a key:

* **De-duplicate on the tight key** — the D-43 fingerprint, which is deliberately conservative
  so that genuinely different houses are never merged.
* **Split on a loose key** — coordinates, rooms and rounded area, ignoring `loc_path`. Any pair
  that *might* be one house then lands wholly in train or wholly in test, and the residual
  cannot leak across the split even though it remains in the sample.

A conservative merge and a conservative split point in opposite directions, and both are cheap.
Stage 5 implements this; the evidence is here (D-60).

## 9 · Two deferred questions, closed

B-20 and D-44 both deferred a judgement to this stage "with the distribution in view". Both are
now in view.

In [ ]:
conflicts = df[df["clean_flags"].astype(str).str.contains("area_quantity_conflict")]
band = df["price_per_marla"].quantile([0.05, 0.95])
inside = conflicts["price_per_marla"].between(band.iloc[0], band.iloc[1])

cheap = df[df["price_per_marla"].between(400_000, 1_000_000)]
cheap_societies = cheap["loc_society"].astype(str).value_counts().head(4).index
society_context = pd.DataFrame({
    "rows in society": [int(df["loc_society"].astype(str).eq(s).sum()) for s in cheap_societies],
    "society median PKR/marla (M)": [
        round(df.loc[df["loc_society"].astype(str).eq(s), "price_per_marla"].median() / 1e6, 2)
        for s in cheap_societies],
}, index=cheap_societies)

print(f"B-20 · area conflicts: {len(conflicts)} rows ({len(conflicts) / len(df):.2%}); "
      f"{int(inside.sum())} of {len(conflicts)} sit inside the corpus 5th-95th percentile "
      f"band under the payload reading")
print(f"       their PKR/marla median {conflicts['price_per_marla'].median() / 1e6:.2f}M "
      f"against a corpus median of {df['price_per_marla'].median() / 1e6:.2f}M")
print(f"\nD-44 · rows between 400k and 1M per marla: {len(cheap)} ({len(cheap) / len(df):.2%}), "
      f"median area {cheap['area_marla'].median():.0f} marla, median price "
      f"{cheap['price'].median() / 1e6:.1f}M")
society_context

**B-20 closes as reviewed, with no change.** Nine rows survive the collapse, 0.14% of the
sample. Under D-32's payload reading their price per marla is indistinguishable from the corpus
— median 3.75M against 3.70M — and seven of nine sit inside the corpus 5th-to-95th percentile
band. Nothing suggests the measured area is the wrong one. D-32's rule stands and the rows stay
flagged. Note the count: B-20 says ten, and nine reach the analysis set.

**D-44's soft boundary stands too, and the reason is in the societies, not the rows.** The 25
rows between 400,000 and 1M per marla sit in developments whose *entire* price level is that
low — Damaan City at 0.80M per marla across all its rows, Barki Road at 1.17M, Chinar Bagh at
1.19M. These are cheap peripheral areas, not mistyped prices, and a tighter band would delete
the bottom of the market rather than clean it. The band stops where "not an asking price" ends,
which is where D-44 put it.

## 10 · Locking the acceptance criteria — B-05, D-47, D-49, D-50

Everything above exists to make these numbers defensible rather than invented. The criteria are
locked against the **D-48 metric**: hold out whole fingerprint groups, predict once per group,
score against every pre-collapse listing at the price its own poster set.

In [ ]:
predicted = baseline_predictions(TIERS["society + phase"])

group_ape = (predicted - df["price"]).abs() / df["price"] * 100
expanded = members.merge(df.assign(p=predicted)[["fp_group_id", "p"]], on="fp_group_id", how="left")
assert len(expanded) == len(members) and expanded["p"].notna().all(), "the D-48 join dropped listings"
listing_ape = (expanded["p"] - expanded["price_listed"]).abs() / expanded["price_listed"] * 100

society_of = df.set_index("fp_group_id")["loc_society"].astype(str)
without_pvv = expanded["fp_group_id"].map(society_of).ne("Park View Villas").to_numpy()

def summarise(ape: pd.Series, label: str) -> dict[str, object]:
    return {"scored on": label, "n": len(ape), "MdAPE %": round(ape.median(), 2),
            "PPE10 %": round(100 * ape.le(10).mean(), 1),
            "PPE20 %": round(100 * ape.le(20).mean(), 1)}

baseline_table = pd.DataFrame([
    summarise(group_ape, "analysis rows (group medians)"),
    summarise(listing_ape, "pre-collapse listings — D-48 headline"),
    summarise(listing_ape[without_pvv], "listings, Park View Villas excluded"),
]).set_index("scored on")
baseline_table

**The collapse gap runs the wrong way, and the reason matters.** The listing-level headline is
*better* than the group-level number, not worse as B-21 assumed. Weighting by listing
over-weights the replicated groups, and those are the easiest rows in the corpus — identical
products in tight price ladders that a locality-and-size median fits well. Take Park View Villas
out and the ordering reverses to what B-21 expected.

That is a live hazard for the criterion, not a curiosity: an acceptance metric scored
listing-level rewards doing well on one developer's bulk inventory. D-48 already requires the
Park-View-excluded number to be reported beside the headline, and that is what guards it.

In [ ]:
CRITERIA = pd.DataFrame([
    {"level": "stretch", "rule": "MdAPE <= 0.80 x baseline", "on today's baseline": "<= 8.50%",
     "and": "PPE20 >= 80%"},
    {"level": "ACCEPT", "rule": "MdAPE <= 0.90 x baseline", "on today's baseline": "<= 9.56%",
     "and": "PPE20 >= 75%, and MdAPE <= 10.0% absolute"},
    {"level": "floor", "rule": "no improvement on the baseline", "on today's baseline": "> 10.62%",
     "and": "reframe the deliverable around the pipeline, the analysis and the baseline"},
]).set_index("level")

SEGMENTS = pd.DataFrame([
    {"segment": "price Q1", "definition": "listing price < 15.5M", "held-out listings": "~416"},
    {"segment": "price Q2", "definition": "15.5M to 25.5M", "held-out listings": "~416"},
    {"segment": "price Q3", "definition": "25.5M to 50.0M", "held-out listings": "~416"},
    {"segment": "price Q4", "definition": "> 50.0M", "held-out listings": "~416"},
    {"segment": "major developments", "definition": "society+phase with >= 100 analysis rows (11 of them)",
     "held-out listings": "~591"},
    {"segment": "mid developments", "definition": "20 to 99 analysis rows (72)", "held-out listings": "~727"},
    {"segment": "long tail", "definition": "under 20 analysis rows (360)", "held-out listings": "~344"},
]).set_index("segment")

print("D-47 — locked levels, measured on the D-48 metric\n")
print(CRITERIA.to_string())
print("\n\nD-50 — segment definitions, fixed before any model exists\n")
print(SEGMENTS.to_string())
print("\n\nD-49 — a segment binds at 1.5x the headline MdAPE threshold, and only above")
print("        100 held-out listings. Every segment above clears that, so all seven bind.")

**Why the margin is relative and not a fixed number of points.** The baseline is itself an
estimate and it will move — a different fold, a size-band change at stage 4, the recency
holdout of D-55 if stage 5 adopts it. A rule reading "10% better than the baseline" means the
same thing after any of those; "1.1 points better" silently gets easier or harder. The absolute
backstop of 10.0% is there so that a *collapsed* baseline cannot make a bad model look good.

**On PPE10.** It is reported and not binding. §2 of `decisions/04-eda.md` records a hard ceiling
of 94.5% on this metric from within-group price variation alone, and a criterion needs a
reachable target; PPE20's ceiling of 98.3% is far enough away not to bind.

**What is *not* locked here.** The optimisation objective stays MAE on log price (§5.2), which
is deliberately not the acceptance metric. The split, the tier and the feature set are stage 4
and 5 decisions on this evidence, per D-52.

## 11 · What this stage decided

Every row below is a number measured in this notebook, and the decision it forced. Nothing
here is a preference: each finding changed something that was already written down, which is
why the decision log is amended rather than appended to in several places.

| Finding | The number | What it decides | Logged as |
|---|---|---|---|
| The simple baseline is already strong | MdAPE 10.62%, PPE10 48.5%, PPE20 72.6%, with no model in it | Acceptance became relative: 0.90 x baseline, PPE20 at least 75%, 10.0% absolute backstop | D-47, closes B-05 |
| Locality signal stops at the phase | Society 12.24%, society plus phase 11.47%, full path 11.50% | Encode at society plus phase. The block tier costs 921 levels and buys nothing | D-52 |
| The window is not a cross section | Median price per marla up 14.1% across twelve months | A recency holdout gains a motive, listing date earns its place as a control | D-55, B-03, B-24 |
| Collapsing duplicates made the metric easier | 10.62% listing level against 11.47% on group medians, reversing to 11.76% without Park View Villas | Validation reports all four figures together, because the headline alone rewards one developer's bulk inventory | D-48, B-21 |
| Some duplication cannot be cleaned away | Upper bound 13.03% of rows | De-duplicate on the strict key, split on a looser one | D-60, B-23 |
| Price per marla rises with plot size | 3.60M at five marla or less, 4.40M at 20 to 40 | Size cannot be banded without location | D-59 |
| Error is not flat across the market | 10.90% in the cheapest price quartile, 15.00% in the dearest | Segmented criteria bind rather than inform, on seven segments fixed in advance | D-49, D-50 |
| Nine columns can never be features | Three leak the target, six are unknowable at inference | The barred list is inherited by stage 4, with the two grounds kept apart | D-51 |
| One development dominates the raw adverts | 807 adverts became 253 rows, 9.7% down to 3.9% | Weighting has less to correct than expected, though bulk adverts price about 9% below individual ones | B-22 |
| Three flagged anomalies dissolved | A size mix artefact, nine area conflicts that price normally, 25 cheap rows in genuinely cheap societies | No further cleaning changes, and one upstream fix | D-58, D-59, D-61 |

The market prices consistently per marla inside a locality and size band. That is why a lookup
table is already a competent estimator, and why the model is measured against that table
rather than against an error target chosen before anyone saw the data.

## 12 · Still to do this stage

* The write-up of B-18, B-19, B-20 and B-22 into BACKLOG, and STATE for the stage close.
* `decisions/04-eda.md`: resolve the remaining stubs from the numbers above.
* Tag `v0.3-eda` and push.